# Credit Risk Decision System
## Stage 2 — Data Understanding

**Project:** Home Credit Default Risk  
**Dataset:** `application_train.csv` (Kaggle)  
**Author:** Portfolio Project  

---

### Objective of This Stage

Before any modelling, we need to understand:
1. What the borrower population looks like
2. Whether the default rate matches real-world lending portfolios
3. Whether income (the traditional indicator) can meaningfully separate defaulters from non-defaulters
4. Which features show real separation — particularly the EXT_SOURCE behavioural signals
5. How to handle the `DAYS_EMPLOYED = 365243` anomaly

This stage either **confirms or challenges** every claim made in Stage 1 (Business Context).  
No assumptions. Numbers only.

---


---
## 1. Imports & Setup

Standard libraries for data manipulation and visualisation.  
`warnings` suppressed to keep output clean.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.facecolor'] = '#FAFAFA'
plt.rcParams['axes.facecolor']   = '#FAFAFA'
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False

# Colour palette: navy = repaid, red = defaulted
C0   = '#2C3E50'   # non-default
C1   = '#E74C3C'   # default
GRID = '#ECF0F1'
BG   = '#FAFAFA'

print("Libraries loaded successfully.")


---
## 2. Load Dataset

We load only `application_train.csv` — the application-level snapshot file.  
This is the only file used in this project (see Stage 1 rationale for why other tables are excluded).

Expected shape: **307,511 rows × 122 columns**


In [ ]:
df = pd.read_csv('application_train.csv')

print(f"Shape: {df.shape}")
print(f"Rows : {df.shape[0]:,}")
print(f"Cols : {df.shape[1]}")


**Confirmation:** 307,511 individual loan applications, each with 122 attributes including the TARGET label.


---
## 3. Target Variable — Default Rate

`TARGET = 1` → borrower defaulted (had difficulty repaying)  
`TARGET = 0` → borrower repaid successfully

This is an **observed outcome**, not a model prediction.

We expect ~8% default rate based on real-world lending portfolios.  
A significantly different number would raise questions about data quality or sampling.


In [ ]:
target_counts = df['TARGET'].value_counts()
default_rate  = df['TARGET'].mean() * 100

print("Target Distribution:")
print(f"  Repaid    (0): {target_counts[0]:>7,}  ({target_counts[0]/len(df)*100:.1f}%)")
print(f"  Defaulted (1): {target_counts[1]:>7,}  ({target_counts[1]/len(df)*100:.1f}%)")
print(f"\nOverall Default Rate: {default_rate:.2f}%")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

bars = ax.bar(['Repaid (0)', 'Defaulted (1)'],
              [target_counts[0], target_counts[1]],
              color=[C0, C1], edgecolor='white', linewidth=1.2, width=0.45)

for bar, count in zip(bars, [target_counts[0], target_counts[1]]):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 3000,
            f'{count:,}\n({count/len(df)*100:.1f}%)',
            ha='center', va='bottom', fontsize=10, fontweight='bold', color=C0)

ax.set_title('Target Distribution — Repaid vs Defaulted', fontweight='bold', fontsize=12)
ax.set_ylabel('Number of Applications')
ax.yaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('plot_01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Observation: Dataset is heavily imbalanced (92% / 8%). This matches real lending portfolios.")


**Insight:**  
The 8.07% default rate confirms this is a realistic lending portfolio dataset.  
The imbalance is not a flaw — it reflects that defaults are rare but costly events, exactly as described in Stage 1.  
This imbalance will be handled at the modelling stage (Stage 3) using `class_weight='balanced'`.


---
## 4. Feature Audit — Our 11 Selected Features

We are working with 11 features out of 122. Before any analysis, we audit:
- Are all features present?
- Do any have missing values that need handling?

The features span three tiers of the business argument:
| Tier | Features |
|------|----------|
| Financial (traditional) | AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY |
| Demographic / Stability | DAYS_EMPLOYED, DAYS_BIRTH, NAME_EDUCATION_TYPE, NAME_INCOME_TYPE, CODE_GENDER |
| Behavioural / External  | EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3 |


In [ ]:
FEATURES = [
    'TARGET',
    # Financial
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
    # Demographic / Stability
    'DAYS_EMPLOYED', 'DAYS_BIRTH',
    'NAME_EDUCATION_TYPE', 'NAME_INCOME_TYPE', 'CODE_GENDER',
    # Behavioural / External
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'
]

df_features = df[FEATURES].copy()

missing = df_features.isnull().sum()
missing_pct = (df_features.isnull().sum() / len(df_features) * 100).round(2)

audit = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print("Feature Audit — Missing Values:")
print(audit.to_string())


**Observations:**

| Feature | Missing | Action |
|---------|---------|--------|
| AMT_ANNUITY | 12 rows | Median imputation — negligible |
| EXT_SOURCE_1 | 173,378 (56%) | Median imputation — high missingness noted |
| EXT_SOURCE_2 | 660 (0.2%) | Median imputation |
| EXT_SOURCE_3 | 60,965 (20%) | Median imputation |
| All others | 0 | No action needed |

EXT_SOURCE_1 has 56% missing. This is a known characteristic of this dataset — the score is simply not available for a large portion of applicants. We impute with median rather than dropping, because dropping would remove over half the dataset.  
Missing pattern will be revisited in Stage 3 (modelling).


---
## 5. Derived Features for Analysis

`DAYS_BIRTH` and `DAYS_EMPLOYED` are stored as **negative integers** (days relative to application date).  
We convert them to interpretable positive units for EDA.

We also flag the `DAYS_EMPLOYED = 365243` anomaly — this will be analysed in Section 8.


In [ ]:
# Age in years (DAYS_BIRTH is negative — distance from today to birth)
df['AGE_YEARS'] = (-df['DAYS_BIRTH'] / 365).round(1)

# Employment duration — replace 365243 with NaN before converting
# 365243 is a special encoded value, NOT actual employment length
df['DAYS_EMPLOYED_CLEAN'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
df['EMPLOYED_YEARS'] = (-df['DAYS_EMPLOYED_CLEAN'] / 365).round(1)

# Binary flag for the 365243 anomaly
df['EMPLOYED_FLAG_SPECIAL'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)

print(f"Age range: {df['AGE_YEARS'].min():.0f} — {df['AGE_YEARS'].max():.0f} years")
print(f"Employment range (excl. flag): {df['EMPLOYED_YEARS'].min():.1f} — {df['EMPLOYED_YEARS'].max():.1f} years")
print(f"\nSpecial employment flag count: {df['EMPLOYED_FLAG_SPECIAL'].sum():,} ({df['EMPLOYED_FLAG_SPECIAL'].mean()*100:.1f}% of applicants)")


---
## 6. Income Analysis — Testing Stage 1's Core Claim

**Stage 1 claimed:** Income is a weak predictor because borrowers with similar income exhibit very different default behaviour.

We test this directly by comparing income distributions of defaulters vs non-defaulters.

If income is a strong separator: the distributions should be clearly distinct.  
If income is a weak separator: the distributions will overlap heavily.


In [ ]:
default_group    = df[df['TARGET'] == 1]
nondefault_group = df[df['TARGET'] == 0]

print("=== Income: Defaulters vs Non-Defaulters ===")
print(f"{'Metric':<20} {'Defaulters':>15} {'Non-Defaulters':>15} {'Difference':>12}")
print("-" * 65)

for metric, func in [('Mean', 'mean'), ('Median', 'median'), ('Std Dev', 'std')]:
    d1_val = getattr(default_group['AMT_INCOME_TOTAL'],    func)()
    d0_val = getattr(nondefault_group['AMT_INCOME_TOTAL'], func)()
    diff   = abs(d1_val - d0_val)
    print(f"{metric:<20} {d1_val:>15,.0f} {d0_val:>15,.0f} {diff:>12,.0f}")

mean_diff_pct = abs(default_group['AMT_INCOME_TOTAL'].mean() - nondefault_group['AMT_INCOME_TOTAL'].mean()) / nondefault_group['AMT_INCOME_TOTAL'].mean() * 100
print(f"\nMean income difference: {mean_diff_pct:.1f}%")
print("→ A 2% difference in means confirms near-complete overlap. Income cannot separate risk groups.")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

cap = 500_000  # cap for readability; extreme outliers exist above this
ax.hist(nondefault_group['AMT_INCOME_TOTAL'].clip(upper=cap), bins=60,
        alpha=0.6, color=C0, label=f'Repaid (n={len(nondefault_group):,})', density=True)
ax.hist(default_group['AMT_INCOME_TOTAL'].clip(upper=cap), bins=60,
        alpha=0.6, color=C1, label=f'Defaulted (n={len(default_group):,})', density=True)

ax.axvline(nondefault_group['AMT_INCOME_TOTAL'].mean(), color=C0,
           linestyle='--', linewidth=1.5, label=f"Non-default mean: {nondefault_group['AMT_INCOME_TOTAL'].mean():,.0f}")
ax.axvline(default_group['AMT_INCOME_TOTAL'].mean(), color=C1,
           linestyle='--', linewidth=1.5, label=f"Default mean: {default_group['AMT_INCOME_TOTAL'].mean():,.0f}")

ax.set_title('Income Distribution — Defaulters vs Non-Defaulters\n(Capped at 500,000 for readability)', fontweight='bold')
ax.set_xlabel('Annual Income')
ax.set_ylabel('Density')
ax.legend(fontsize=9)
ax.text(0.97, 0.92, 'Mean difference: 2.0%\nConclusion: Income cannot separate risk',
        transform=ax.transAxes, ha='right', fontsize=9, color='gray',
        style='italic', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
plt.tight_layout()
plt.savefig('plot_02_income_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


**Finding:** Mean income differs by only **2.0%** between defaulters and non-defaulters.  
The distributions are visually inseparable. This directly validates Stage 1's claim:  
> *"Income reflects earning capacity. It does not capture the signals that predict repayment behaviour."*


---
## 7. EXT_SOURCE Analysis — Behavioural Signal Strength

EXT_SOURCE_1/2/3 are external credit bureau / behavioural risk scores.  
They are opaque composites — Home Credit does not publish their exact formula —  
but they are known to encode past repayment behaviour, delinquency, and credit utilisation.

Stage 1 argued these are the signals income cannot see.  
We now measure how well they separate defaulters from non-defaulters.

**Metric used:** Gap between group means (higher = better separation).  
Compare to income's gap of ~3,466 on a base of ~169,000 (2.0%).


In [ ]:
print(f"{'Feature':<15} {'Default Mean':>14} {'Non-Default Mean':>18} {'Gap':>10} {'Gap %':>8}")
print("-" * 70)

for col in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']:
    d1_mean = default_group[col].dropna().mean()
    d0_mean = nondefault_group[col].dropna().mean()
    gap     = abs(d1_mean - d0_mean)
    gap_pct = gap / d0_mean * 100
    print(f"{col:<15} {d1_mean:>14.4f} {d0_mean:>18.4f} {gap:>10.4f} {gap_pct:>7.1f}%")

print("\n→ Compare to income gap of 2.0%. EXT_SOURCE gaps are ~24-26% of the mean — 12x stronger separation.")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

gaps = {'EXT_SOURCE_1': 0.1245, 'EXT_SOURCE_2': 0.1125, 'EXT_SOURCE_3': 0.1303}

for ax, col in zip(axes, ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']):
    d0 = nondefault_group[col].dropna()
    d1 = default_group[col].dropna()

    ax.hist(d0, bins=50, alpha=0.6, color=C0,
            label=f'Repaid (n={len(d0):,})',    density=True)
    ax.hist(d1, bins=50, alpha=0.6, color=C1,
            label=f'Defaulted (n={len(d1):,})', density=True)

    ax.axvline(d0.mean(), color=C0, linestyle='--', linewidth=1.5,
               label=f'Non-default mean: {d0.mean():.3f}')
    ax.axvline(d1.mean(), color=C1, linestyle='--', linewidth=1.5,
               label=f'Default mean: {d1.mean():.3f}')

    ax.set_title(f'{col}\nSeparation Gap: {gaps[col]:.4f}', fontweight='bold')
    ax.set_xlabel('Score Value')
    ax.set_ylabel('Density')
    ax.legend(fontsize=7)

fig.suptitle('EXT_SOURCE Variables — Behavioural Signal Separation\nLower scores = higher default risk',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_03_ext_source_separation.png', dpi=150, bbox_inches='tight')
plt.show()


**Finding:**  
All three EXT_SOURCE variables show a consistent, visually clear separation between defaulters and non-defaulters.

- EXT_SOURCE_3 has the strongest gap: **0.1303**  
- EXT_SOURCE_1 follows: **0.1245**  
- EXT_SOURCE_2: **0.1125**

Defaulters consistently score **lower** on all three — lower external risk scores predict higher default probability.  
This is approximately 12× stronger separation than income — measured as relative gap between group means (24% vs 2%).  

This confirms the core hypothesis of the project: **behavioural signals outperform traditional financial indicators.**


---
## 8. DAYS_EMPLOYED = 365243 — Anomaly Analysis

`DAYS_EMPLOYED` stores employment duration as a negative integer (days before application).  
The value `365243` is a special encoded value — it does NOT represent ~1,000 years of employment.

This anomaly affects **18% of the dataset** and must be handled correctly.

**Two wrong approaches:**
1. Drop rows → lose 18% of data
2. Use raw value → corrupt the model with a meaningless outlier

**Correct approach:**
- Replace 365243 with `NaN` for the continuous employment feature
- Create a **binary flag** (`EMPLOYED_FLAG_SPECIAL = 1`) to preserve the information

Before doing that, we analyse: do flagged borrowers default at a different rate?  
This tells us whether the flag itself carries signal.


In [ ]:
flagged     = df[df['DAYS_EMPLOYED'] == 365243]
not_flagged = df[df['DAYS_EMPLOYED'] != 365243]

flag_default_rate     = flagged['TARGET'].mean() * 100
nonflag_default_rate  = not_flagged['TARGET'].mean() * 100
overall_default_rate  = df['TARGET'].mean() * 100

print(f"{'Group':<35} {'Count':>8} {'Default Rate':>14}")
print("-" * 60)
print(f"{'Active Employment (365243 absent)':<35} {len(not_flagged):>8,} {nonflag_default_rate:>13.2f}%")
print(f"{'Special Flag = 365243':<35} {len(flagged):>8,} {flag_default_rate:>13.2f}%")
print(f"{'Overall':<35} {len(df):>8,} {overall_default_rate:>13.2f}%")
print()
print(f"Flagged borrowers default at {flag_default_rate:.2f}% vs {nonflag_default_rate:.2f}% for active employment.")
print(f"This is LOWER than average — the flag likely encodes pensioners/retirees, not unemployed.")
print(f"The flag carries real signal: it should be retained as a binary feature.")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

groups = ['Active\nEmployment\n(82%)', 'Special Flag\n(18%)\n[Pensioner/Unknown]', 'Overall\nAverage']
rates  = [nonflag_default_rate, flag_default_rate, overall_default_rate]
colors = [C1, C0, 'gray']

bars = ax.bar(groups, rates, color=colors, edgecolor='white', linewidth=1.2, width=0.45)
ax.axhline(overall_default_rate, color='gray', linestyle='--', linewidth=1,
           label=f'Overall avg: {overall_default_rate:.1f}%')

for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{rate:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_title('DAYS_EMPLOYED = 365243\nDefault Rate by Flag Status', fontweight='bold')
ax.set_ylabel('Default Rate (%)')
ax.legend(fontsize=9)
ax.yaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('plot_04_employment_flag.png', dpi=150, bbox_inches='tight')
plt.show()


**Finding:**  
Flagged borrowers (365243) default at **5.40%** — significantly lower than the 8.66% for borrowers with recorded employment.

This rules out the "unemployed" interpretation. The flag most likely encodes **pensioners, retirees, or self-employed** borrowers who don't have a standard employment record.

**Modelling decision:**
- `DAYS_EMPLOYED_CLEAN`: replace 365243 with NaN, then median-impute
- `EMPLOYED_FLAG_SPECIAL`: binary (0/1) — retained as a separate feature to preserve this signal

Treating this value as standard missing data without preserving the flag would remove a meaningful risk signal and introduce systematic bias into the model.


---
## 9. Age Analysis

Age (`DAYS_BIRTH` converted to years) is a demographic stability signal.  
Younger borrowers are typically earlier in their financial life — lower savings, less stable income history.

We test whether age shows meaningful separation.


In [ ]:
print("=== Age: Defaulters vs Non-Defaulters ===")
print(f"Defaulters     — Mean age: {default_group['AGE_YEARS'].mean():.1f} yrs, Median: {default_group['AGE_YEARS'].median():.1f} yrs")
print(f"Non-Defaulters — Mean age: {nondefault_group['AGE_YEARS'].mean():.1f} yrs, Median: {nondefault_group['AGE_YEARS'].median():.1f} yrs")
print(f"Mean difference: {abs(default_group['AGE_YEARS'].mean() - nondefault_group['AGE_YEARS'].mean()):.1f} years")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(nondefault_group['AGE_YEARS'], bins=40, alpha=0.6, color=C0,
        label=f'Repaid (mean: {nondefault_group["AGE_YEARS"].mean():.1f} yrs)', density=True)
ax.hist(default_group['AGE_YEARS'],    bins=40, alpha=0.6, color=C1,
        label=f'Defaulted (mean: {default_group["AGE_YEARS"].mean():.1f} yrs)', density=True)

ax.axvline(nondefault_group['AGE_YEARS'].mean(), color=C0, linestyle='--', linewidth=1.5)
ax.axvline(default_group['AGE_YEARS'].mean(),    color=C1, linestyle='--', linewidth=1.5)

ax.set_title('Age Distribution — Defaulters vs Non-Defaulters', fontweight='bold')
ax.set_xlabel('Age (years)')
ax.set_ylabel('Density')
ax.legend(fontsize=9)
ax.text(0.97, 0.90, 'Default peak clusters in 30s\nYounger = higher risk tendency',
        transform=ax.transAxes, ha='right', fontsize=9, color='gray', style='italic',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
plt.tight_layout()
plt.savefig('plot_05_age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


**Finding:**  
Mean age of defaulters: **40.8 years** vs non-defaulters: **44.2 years** — a 3.4-year gap.  
The default distribution peaks in the 30s, consistent with early-career financial instability.  
Age has moderate signal — weaker than EXT_SOURCE but stronger than income alone.


---
## 10. Categorical Features — Default Rates by Group

We analyse three categorical features:
- `NAME_EDUCATION_TYPE` — education level as a socioeconomic proxy
- `NAME_INCOME_TYPE` — employment category as a stability proxy
- `CODE_GENDER` — gender (statistically significant in this dataset)

For each, we compute the default rate per category and compare to the overall 8.07% baseline.


In [ ]:
print("=== Default Rate by Education Type ===")
ed = (df.groupby('NAME_EDUCATION_TYPE')['TARGET']
        .agg(['mean', 'count'])
        .sort_values('mean', ascending=False))
ed['default_rate_%'] = (ed['mean'] * 100).round(2)
ed['share_%'] = (ed['count'] / len(df) * 100).round(1)
print(ed[['default_rate_%', 'count', 'share_%']].to_string())
print(f"\nBaseline: {df['TARGET'].mean()*100:.2f}%")


In [ ]:
print("=== Default Rate by Income Type (main categories) ===")
main_income_types = ['Working', 'Commercial associate', 'State servant', 'Pensioner']
inc = (df[df['NAME_INCOME_TYPE'].isin(main_income_types)]
         .groupby('NAME_INCOME_TYPE')['TARGET']
         .agg(['mean', 'count'])
         .sort_values('mean', ascending=False))
inc['default_rate_%'] = (inc['mean'] * 100).round(2)
inc['share_%'] = (inc['count'] / len(df) * 100).round(1)
print(inc[['default_rate_%', 'count', 'share_%']].to_string())

print("\n=== Default Rate by Gender ===")
gen = (df[df['CODE_GENDER'].isin(['M','F'])]
         .groupby('CODE_GENDER')['TARGET']
         .agg(['mean', 'count'])
         .sort_values('mean', ascending=False))
gen['default_rate_%'] = (gen['mean'] * 100).round(2)
print(gen[['default_rate_%', 'count']].to_string())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Education
short_labels = {
    'Lower secondary':               'Lower Sec.',
    'Secondary / secondary special': 'Secondary',
    'Incomplete higher':             'Incomplete Higher',
    'Higher education':              'Higher Education',
    'Academic degree':               'Academic Degree'
}
ed_plot = ed.copy()
ed_plot.index = [short_labels.get(x, x) for x in ed_plot.index]
ed_plot = ed_plot.sort_values('default_rate_%', ascending=True)
colors_ed = [C1 if v > df['TARGET'].mean()*100 else C0 for v in ed_plot['default_rate_%']]
bars = axes[0].barh(ed_plot.index, ed_plot['default_rate_%'], color=colors_ed, edgecolor='white')
axes[0].axvline(df['TARGET'].mean()*100, color='gray', linestyle='--', linewidth=1, label='Avg 8.1%')
axes[0].set_title('Default Rate\nby Education Type', fontweight='bold')
axes[0].set_xlabel('Default Rate (%)')
axes[0].legend(fontsize=8)
for bar, val in zip(bars, ed_plot['default_rate_%']):
    axes[0].text(val + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=8, fontweight='bold')

# Income type
inc_plot = inc.sort_values('default_rate_%', ascending=True)
colors_inc = [C1 if v > df['TARGET'].mean()*100 else C0 for v in inc_plot['default_rate_%']]
bars2 = axes[1].barh(inc_plot.index, inc_plot['default_rate_%'], color=colors_inc, edgecolor='white')
axes[1].axvline(df['TARGET'].mean()*100, color='gray', linestyle='--', linewidth=1, label='Avg 8.1%')
axes[1].set_title('Default Rate\nby Income Type', fontweight='bold')
axes[1].set_xlabel('Default Rate (%)')
axes[1].legend(fontsize=8)
for bar, val in zip(bars2, inc_plot['default_rate_%']):
    axes[1].text(val + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=8, fontweight='bold')

# Gender
gen_main = gen[gen.index.isin(['M','F'])].sort_values('default_rate_%', ascending=True)
colors_gen = [C1 if v > df['TARGET'].mean()*100 else C0 for v in gen_main['default_rate_%']]
bars3 = axes[2].bar(gen_main.index, gen_main['default_rate_%'], color=colors_gen,
                    edgecolor='white', width=0.35)
axes[2].axhline(df['TARGET'].mean()*100, color='gray', linestyle='--', linewidth=1, label='Avg 8.1%')
axes[2].set_title('Default Rate\nby Gender', fontweight='bold')
axes[2].set_ylabel('Default Rate (%)')
axes[2].legend(fontsize=8)
for bar, val in zip(bars3, gen_main['default_rate_%']):
    axes[2].text(bar.get_x() + bar.get_width()/2, val + 0.1,
                 f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.suptitle('Categorical Features — Default Rate by Group', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_06_categorical_default_rates.png', dpi=150, bbox_inches='tight')
plt.show()


**Findings:**

**Education:** Clear gradient — lower education correlates with higher default rate.  
Lower secondary: **10.93%** vs Academic degree: **1.83%**. Range of 9.1pp above a baseline of 8.1%.

**Income type:** Working borrowers (9.59%) default above average; Pensioners (5.39%) and State servants (5.75%) well below.  
Employment stability and income predictability are real signals.

**Gender:** Male borrowers default at **10.14%** vs female at **7.00%**.  
A statistically meaningful difference given the sample sizes (105k male, 202k female).


---
## 11. Credit Amount — Revenue Proxy Verification

`AMT_CREDIT` is the total loan amount requested. In Stage 5 (Policy Simulation),  
we use `sum(AMT_CREDIT)` as the **revenue proxy** — the business volume metric against which  
default reduction is traded off.

We verify the distribution here to ensure it's a sensible proxy.


In [ ]:
print("=== AMT_CREDIT: Defaulters vs Non-Defaulters ===")
for metric, func in [('Mean', 'mean'), ('Median', 'median'), ('Total (sum)', 'sum')]:
    d1 = getattr(default_group['AMT_CREDIT'], func)()
    d0 = getattr(nondefault_group['AMT_CREDIT'], func)()
    print(f"{metric:<20} Defaulters: {d1:>14,.0f}   Non-Defaulters: {d0:>14,.0f}")

total_credit = df['AMT_CREDIT'].sum()
default_credit = default_group['AMT_CREDIT'].sum()
print(f"\nTotal portfolio credit: {total_credit:,.0f}")
print(f"Credit at risk (defaulters): {default_credit:,.0f} ({default_credit/total_credit*100:.1f}% of total)")
print("\n→ This confirms AMT_CREDIT sum is a valid revenue proxy for policy simulation.")


---
## 12. Stage 2 Summary — Key Findings

| # | Finding | Implication |
|---|---------|-------------|
| 1 | Default rate: **8.07%** | Realistic imbalanced portfolio — confirmed |
| 2 | Income mean difference: **2.0%** | Income cannot separate defaulters — Stage 1 confirmed |
| 3 | EXT_SOURCE gaps: **0.11–0.13** | ~12× stronger than income — behavioural signals work |
| 4 | Age gap: **3.4 years** | Moderate signal — younger borrowers higher risk |
| 5 | Education range: **1.83%–10.93%** | Strong categorical signal |
| 6 | 365243 flag default rate: **5.40%** | Flag = pensioner/retiree, not unemployed — must be retained |
| 7 | EXT_SOURCE_1 missing: **56%** | High missingness — median impute, not drop |

---

##### *Across financial and demographic features, separation between defaulters and non-defaulters remains limited, confirming that traditional indicators alone are insufficient for risk assessment. In contrast, behavioural signals captured through EXT_SOURCE variables show clear and consistent differentiation, validating the need for a probability-based decision framework.*

### Analytical Direction for Stage 3

The data has confirmed the business hypothesis. We now move to modelling:

- Train a **Logistic Regression** on the 11 selected features
- Handle missingness (median imputation) and the 365243 flag
- Interpret **coefficients** to formally rank feature importance
- Validate that EXT_SOURCE variables dominate the model

**→ Stage 3: Risk Driver Analysis & Model**
